In [1]:
import cv2
import numpy as np
import sys

In [3]:
cam = cv2.VideoCapture(0)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 320)

face = cv2.CascadeClassifier(r"C:\Users\monukoru\Downloads\image processing python\DATA\haarcascades\haarcascade_frontalface_alt.xml")

cv2.namedWindow('track', cv2.WINDOW_NORMAL)

# Initial face detection 
while True:
    ret, frame = cam.read()
    if not ret:
        sys.exit("Failed to read camera frame")
    frame = cv2.flip(frame, 1)

    small_gray = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (0, 0), fx=0.5, fy=0.5)
    dimension = face.detectMultiScale(small_gray)

    if len(dimension) == 0:
        cv2.imshow('track', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            cam.release()
            cv2.destroyAllWindows()
            sys.exit()
        continue
    else:
        break

# scaling back
x_fc, y_fc, w_fc, h_fc = tuple(dimension[0] << 1)
track_window = (x_fc, y_fc, w_fc, h_fc)

roi = frame[y_fc:y_fc + h_fc, x_fc:x_fc + w_fc]
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# Skin color 
H, S, V = hsv_roi[:, :, 0].flatten(), hsv_roi[:, :, 1].flatten(), hsv_roi[:, :, 2].flatten()
h_lower, h_upper = np.percentile(H, [20, 80])
s_lower, s_upper = np.percentile(S, [20, 80])
v_lower, v_upper = np.percentile(V, [20, 80])
lower_skin = np.array([h_lower, s_lower, v_lower], dtype=np.uint8)
upper_skin = np.array([h_upper, s_upper, v_upper], dtype=np.uint8)

roi_hist = cv2.calcHist([hsv_roi], [2], None, [256], [0, 256])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)

#  tracking 
while True:
    ret, frame = cam.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)

    hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    back_proj = cv2.calcBackProject([hsv_frame], [0], roi_hist, [0, 256], 1)

    _, track_window = cv2.meanShift(back_proj, track_window, term_crit)

    skin_mask_full = cv2.inRange(hsv_frame, lower_skin, upper_skin)
    x, y, w, h = track_window
    x, y = max(0, x), max(0, y)
    w, h = min(w, frame.shape[1] - x), min(h, frame.shape[0] - y)
    roi_mask = skin_mask_full[y:y + h, x:x + w]
    skin_ratio = cv2.countNonZero(roi_mask) / (w * h) if w * h > 0 else 0   # that function used calculates the skin ratio where it creates a mask, where 255 is skin color and 0
    # is some other color, then it divides those no. of pixels with actual no. of pixels present at the tracking

    if skin_ratio < 0.5:  # when box goes away
        small_gray = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (0, 0), fx=0.5, fy=0.5)
        dimension = face.detectMultiScale(small_gray)

        if len(dimension) > 0:
            x_fc, y_fc, w_fc, h_fc = tuple(dimension[0] << 1)
            track_window = (x_fc, y_fc, w_fc, h_fc)

            roi = frame[y_fc:y_fc + h_fc, x_fc:x_fc + w_fc]
            hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

            H, S, V = hsv_roi[:, :, 0].flatten(), hsv_roi[:, :, 1].flatten(), hsv_roi[:, :, 2].flatten()
            h_lower, h_upper = np.percentile(H, [20, 80])
            s_lower, s_upper = np.percentile(S, [20, 80])
            v_lower, v_upper = np.percentile(V, [20, 80])
            lower_skin = np.array([h_lower, s_lower, v_lower], dtype=np.uint8)
            upper_skin = np.array([h_upper, s_upper, v_upper], dtype=np.uint8)

            roi_hist = cv2.calcHist([hsv_roi], [2], None, [256], [0, 256])
            cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

    x, y, w, h = track_window
    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow('track', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()